# 🤖 DeepSeek AI Crypto Analysis with Binance Data
## Interactive Crypto Query with AI-Powered Analysis

This notebook provides:
- **AI-powered ticker extraction** from natural language queries
- **Real-time price data** from Binance API (with yfinance fallback)
- **Technical analysis** with 32+ indicators
- **DeepSeek AI analysis** for trading recommendations
- **Complete pipeline** from query to actionable insights

## 1. Setup and Imports

In [1]:
# Core imports
import os
import sys
import warnings
import re
from datetime import datetime
warnings.filterwarnings('ignore')

# Data science libraries
import pandas as pd
import numpy as np
import yfinance as yf

# AI and environment
import openai
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✅ Core libraries imported successfully!")
print(f"📅 Setup Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Core libraries imported successfully!
📅 Setup Date: 2025-09-03 16:55:40


In [2]:
# Import our crypto analysis modules
try:
    from crypto_ta_simple import get_crypto_data, validate_data, TechnicalAnalyzer
    print("✅ Technical analysis functions loaded")
    TA_AVAILABLE = True
except ImportError as e:
    print(f"⚠️  Technical analysis not available: {e}")
    TA_AVAILABLE = False

# Import Binance API if available
try:
    from binance_api import BinanceAPI
    binance_client = BinanceAPI()
    BINANCE_AVAILABLE = True
    print("✅ Binance API module loaded")
except ImportError:
    BINANCE_AVAILABLE = False
    binance_client = None
    print("⚠️  Binance API not available, using yfinance fallback")

print("🚀 Module imports complete!")

🔧 Installing required packages...
✅ pandas already installed
✅ numpy already installed
✅ matplotlib already installed
✅ seaborn already installed
✅ plotly already installed
✅ ta already installed
✅ yfinance already installed
✅ Custom Binance API module imported

🎉 All dependencies loaded successfully!
📅 Analysis Date: 2025-09-03 16:55:40

🚀 Configuration Set:
   Symbol: BTC
   Period: 3mo
   Interval: 1d
✅ TechnicalAnalyzer class defined successfully!

🚀 Starting Technical Analysis for BTC
📊 Period: 3mo, Interval: 1d

1️⃣ Fetching market data...
📊 Fetching BTC data from Binance API...
✅ Successfully fetched 90 records from Binance
✅ Successfully loaded 90 records
📅 Date range: 2025-06-06 to 2025-09-03
💰 Current price: $111,060.73

2️⃣ Calculating technical indicators...
🔄 Calculating technical indicators...
✅ Moving averages calculated
✅ Trend indicators calculated
✅ Momentum indicators calculated
✅ Volatility indicators calculated
✅ Volume indicators calculated
✅ All technical indicat

## 2. AI Ticker Extraction

In [3]:
def extract_ticker_with_ai(query):
    """
    Use DeepSeek AI to extract cryptocurrency ticker from natural language query.
    """
    api_key = os.getenv('OPENROUTER_API_KEY')
    if not api_key:
        print("⚠️  OPENROUTER_API_KEY not found in environment variables")
        print("💡 Add OPENROUTER_API_KEY=your_key_here to your .env file")
        return None
    
    try:
        client = openai.OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key
        )
        
        prompt = f"""
Extract the cryptocurrency ticker symbol from this query: "{query}"

Rules:
- Return ONLY the ticker symbol (e.g., BTC, ETH, ADA, SOL, MATIC)
- If multiple cryptos mentioned, return the main one
- If no crypto found, return "UNKNOWN"
- Common mappings: Bitcoin=BTC, Ethereum=ETH, Cardano=ADA, Solana=SOL, Polygon=MATIC

Examples:
"How is Bitcoin doing?" → BTC
"Should I buy Ethereum?" → ETH
"What about Cardano price?" → ADA

Query: "{query}"
Ticker:
"""
        
        response = client.chat.completions.create(
            model="deepseek/deepseek-chat-v3.1:free",
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=50,
            temperature=0.1
        )
        
        ticker = response.choices[0].message.content.strip().upper()
        
        # Clean up the response
        ticker = re.sub(r'[^A-Z]', '', ticker)
        
        if ticker and ticker != "UNKNOWN":
            return ticker
        else:
            return None
            
    except Exception as e:
        print(f"❌ AI ticker extraction failed: {e}")
        return None

print("🎯 AI Ticker Extraction Function Ready!")

🎯 AI Ticker Extraction Function Ready!


## 3. Price Data Functions

In [4]:
def get_binance_price_data(ticker):
    """
    Get current price and basic data from Binance API with yfinance fallback.
    """
    if not BINANCE_AVAILABLE:
        # Fallback to yfinance
        try:
            yf_ticker = yf.Ticker(f"{ticker}-USD")
            hist = yf_ticker.history(period="5d")
            
            if not hist.empty:
                current_price = hist['Close'].iloc[-1]
                prev_price = hist['Close'].iloc[-2] if len(hist) > 1 else current_price
                change_24h = ((current_price - prev_price) / prev_price) * 100
                
                return {
                    'success': True,
                    'price': current_price,
                    'change_24h': change_24h,
                    'volume': hist['Volume'].iloc[-1] if 'Volume' in hist.columns else 0,
                    'source': 'yfinance'
                }
        except Exception as e:
            return {'success': False, 'error': f"YFinance error: {e}"}
    
    try:
        # Use Binance API
        result = binance_client.get_current_price(ticker)
        if result.get('success'):
            # Binance API returns data directly in result, not in result['data']
            return {
                'success': True,
                'price': float(result['current_price']),
                'change_24h': float(result.get('price_change_24h', 0)),
                'volume': float(result.get('volume_24h', 0)),
                'source': 'binance'
            }
        else:
            return {'success': False, 'error': result.get('error', 'Unknown Binance error')}
    except Exception as e:
        return {'success': False, 'error': f"Binance API error: {e}"}

print("💰 Price Data Function Ready!")

💰 Price Data Function Ready!


## 4. AI Analysis Function

In [5]:
def analyze_with_deepseek(ticker, price_data, technical_data=None):
    """
    Get AI analysis from DeepSeek about the cryptocurrency.
    """
    api_key = os.getenv('OPENROUTER_API_KEY')
    if not api_key:
        return "⚠️  OpenRouter API key not available for AI analysis"
    
    try:
        client = openai.OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key
        )
        
        # Prepare technical analysis summary if available
        tech_summary = ""
        if technical_data is not None and not technical_data.empty:
            latest = technical_data.iloc[-1]
            # Format technical indicators safely for AI analysis
            rsi_val = latest.get('RSI')
            rsi_str = f"{rsi_val:.2f}" if pd.notna(rsi_val) else 'N/A'
            
            macd_val = latest.get('MACD')
            macd_str = f"{macd_val:.4f}" if pd.notna(macd_val) else 'N/A'
            
            sma20_val = latest.get('SMA_20')
            if pd.notna(sma20_val) and sma20_val > 0:
                sma20_pct = ((latest['Close'] / sma20_val) - 1) * 100
                sma20_str = f"{sma20_pct:+.2f}%"
            else:
                sma20_str = 'N/A'
            
            sma50_val = latest.get('SMA_50')
            if pd.notna(sma50_val) and sma50_val > 0:
                sma50_pct = ((latest['Close'] / sma50_val) - 1) * 100
                sma50_str = f"{sma50_pct:+.2f}%"
            else:
                sma50_str = 'N/A'
            
            tech_summary = f"""
Technical Analysis Summary:
- RSI: {rsi_str}
- MACD: {macd_str}
- Price vs SMA(20): {sma20_str}
- Price vs SMA(50): {sma50_str}
"""
        
        prompt = f"""
Analyze {ticker} cryptocurrency based on the following data:

Price Data:
- Current Price: ${price_data['price']:,.2f}
- 24h Change: {price_data['change_24h']:+.2f}%
- Volume: {price_data['volume']:,.0f}
- Data Source: {price_data['source']}
{tech_summary}

Provide a concise analysis including:
1. Current market sentiment (bullish/bearish/neutral)
2. Key technical signals
3. Trading recommendation (BUY/HOLD/SELL) with confidence level
4. Risk assessment

Keep response under 200 words and focus on actionable insights.
"""
        
        response = client.chat.completions.create(
            model="deepseek/deepseek-chat-v3.1:free",
            messages=[
                {"role": "user", "content": prompt}
            ],
            max_tokens=300,
            temperature=0.3
        )
        
        return response.choices[0].message.content.strip()
        
    except Exception as e:
        return f"❌ AI analysis failed: {e}"

print("🤖 AI Analysis Function Ready!")

🤖 AI Analysis Function Ready!


## 5. Complete Analysis Pipeline

In [6]:
def process_crypto_query_with_ai(user_query):
    """
    Complete pipeline: Extract ticker → Get price data → Get technical analysis → AI analysis
    """
    print(f"🤖 Processing query: '{user_query}'")
    print("=" * 60)
    
    # Step 1: Extract ticker using AI
    print("1️⃣ Extracting cryptocurrency ticker...")
    ticker = extract_ticker_with_ai(user_query)
    
    if not ticker:
        return "❌ Could not identify cryptocurrency from your query. Please mention a specific crypto like Bitcoin, Ethereum, etc."
    
    print(f"   ✅ Identified ticker: {ticker}")
    
    # Step 2: Get price data
    print("\n2️⃣ Fetching current price data...")
    price_result = get_binance_price_data(ticker)
    
    if not price_result['success']:
        return f"❌ Failed to get price data for {ticker}: {price_result['error']}"
    
    price_data = price_result
    print(f"   ✅ Current price: ${price_data['price']:,.2f} ({price_data['change_24h']:+.2f}%)")
    print(f"   📊 Data source: {price_data['source']}")
    
    # Step 3: Get technical analysis (if possible)
    print("\n3️⃣ Performing technical analysis...")
    technical_data = None
    
    if TA_AVAILABLE:
        try:
            # Try to get historical data for technical analysis
            raw_data = get_crypto_data(ticker, period='1mo', interval='1d')
            if raw_data is not None and not raw_data.empty:
                validated_data = validate_data(raw_data)
                if validated_data is not None and not validated_data.empty:
                    analyzer = TechnicalAnalyzer(validated_data)
                    technical_data = analyzer.calculate_all_indicators()
                    print(f"   ✅ Technical analysis completed ({len(technical_data)} records)")
                else:
                    print("   ⚠️  Data validation failed")
            else:
                print("   ⚠️  Could not fetch historical data")
        except Exception as e:
            print(f"   ⚠️  Technical analysis failed: {e}")
    else:
        print("   ⚠️  Technical analysis module not available")
    
    # Step 4: Get AI analysis
    print("\n4️⃣ Generating AI analysis...")
    ai_analysis = analyze_with_deepseek(ticker, price_data, technical_data)
    
    # Step 5: Format final result
    print("\n" + "=" * 60)
    print(f"📊 ANALYSIS REPORT FOR {ticker}")
    print("=" * 60)
    
    result = f"""
💰 **PRICE DATA**
Current Price: ${price_data['price']:,.2f}
24h Change: {price_data['change_24h']:+.2f}%
Volume: {price_data['volume']:,.0f}
Data Source: {price_data['source'].title()}

📈 **TECHNICAL INDICATORS**
"""
    
    if technical_data is not None and not technical_data.empty:
        latest = technical_data.iloc[-1]
        # Format technical indicators safely
        rsi_val = latest.get('RSI')
        rsi_str = f"{rsi_val:.2f}" if pd.notna(rsi_val) else 'N/A'
        
        macd_val = latest.get('MACD')
        macd_str = f"{macd_val:.4f}" if pd.notna(macd_val) else 'N/A'
        
        sma20_val = latest.get('SMA_20')
        if pd.notna(sma20_val) and sma20_val > 0:
            sma20_pct = ((latest['Close'] / sma20_val) - 1) * 100
            sma20_str = f"{sma20_pct:+.2f}%"
        else:
            sma20_str = 'N/A'
        
        sma50_val = latest.get('SMA_50')
        if pd.notna(sma50_val) and sma50_val > 0:
            sma50_pct = ((latest['Close'] / sma50_val) - 1) * 100
            sma50_str = f"{sma50_pct:+.2f}%"
        else:
            sma50_str = 'N/A'
        
        result += f"""
RSI (14): {rsi_str}
MACD: {macd_str}
Price vs SMA(20): {sma20_str}
Price vs SMA(50): {sma50_str}
"""
    else:
        result += "\nTechnical analysis not available"
    
    result += f"""

🤖 **AI ANALYSIS**
{ai_analysis}

📅 Analysis Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
"""
    
    print(result)
    return result

print("🚀 Complete AI Analysis Pipeline Ready!")
print("💡 Use process_crypto_query_with_ai('your question here') to analyze any crypto")

🚀 Complete AI Analysis Pipeline Ready!
💡 Use process_crypto_query_with_ai('your question here') to analyze any crypto


## 6. 🎯 Interactive Query Interface
### Enter your crypto query and get AI-powered analysis!

In [7]:
# 🎯 ENTER YOUR CRYPTO QUERY HERE
# Examples:
# - "How is Bitcoin performing today?"
# - "Should I buy Ethereum now?"
# - "What's happening with Solana?"
# - "Is Cardano a good investment?"
# - "Tell me about Polygon MATIC"

USER_QUERY = "How is Bitcoin performing today?"  # 👈 CHANGE THIS TO YOUR QUESTION

# Process the query
if USER_QUERY.strip():
    result = process_crypto_query_with_ai(USER_QUERY)
else:
    print("💡 Please enter a crypto query in the USER_QUERY variable above")
    print("\nExamples:")
    print('USER_QUERY = "How is Bitcoin doing?"')
    print('USER_QUERY = "Should I buy Ethereum?"')
    print('USER_QUERY = "What about Solana price?"')

🤖 Processing query: 'How is Bitcoin performing today?'
1️⃣ Extracting cryptocurrency ticker...
   ✅ Identified ticker: BTC

2️⃣ Fetching current price data...
   ✅ Current price: $111,060.73 (+2.25%)
   📊 Data source: binance

3️⃣ Performing technical analysis...
📊 Fetching BTC data from Binance API...
✅ Successfully fetched 30 records from Binance
🔄 Calculating technical indicators...
✅ Moving averages calculated
✅ Trend indicators calculated
✅ Momentum indicators calculated
✅ Volatility indicators calculated
✅ Volume indicators calculated
✅ All technical indicators calculated successfully!
   ✅ Technical analysis completed (30 records)

4️⃣ Generating AI analysis...

📊 ANALYSIS REPORT FOR BTC

💰 **PRICE DATA**
Current Price: $111,060.73
24h Change: +2.25%
Volume: 15,440
Data Source: Binance

📈 **TECHNICAL INDICATORS**

RSI (14): 45.17
MACD: -1581.3268
Price vs SMA(20): -1.57%
Price vs SMA(50): N/A


🤖 **AI ANALYSIS**
Based on the provided data, the market sentiment is **neutral to sl

## 7. 🔧 Quick Test Examples
### Run these cells to test different cryptocurrencies

In [8]:
# Test Bitcoin
process_crypto_query_with_ai("How is Bitcoin doing today?")

🤖 Processing query: 'How is Bitcoin doing today?'
1️⃣ Extracting cryptocurrency ticker...
   ✅ Identified ticker: BTC

2️⃣ Fetching current price data...
   ✅ Current price: $111,034.20 (+2.22%)
   📊 Data source: binance

3️⃣ Performing technical analysis...
📊 Fetching BTC data from Binance API...
✅ Successfully fetched 30 records from Binance
🔄 Calculating technical indicators...
✅ Moving averages calculated
✅ Trend indicators calculated
✅ Momentum indicators calculated
✅ Volatility indicators calculated
✅ Volume indicators calculated
✅ All technical indicators calculated successfully!
   ✅ Technical analysis completed (30 records)

4️⃣ Generating AI analysis...

📊 ANALYSIS REPORT FOR BTC

💰 **PRICE DATA**
Current Price: $111,034.20
24h Change: +2.22%
Volume: 15,434
Data Source: Binance

📈 **TECHNICAL INDICATORS**

RSI (14): 45.11
MACD: -1583.4439
Price vs SMA(20): -1.59%
Price vs SMA(50): N/A


🤖 **AI ANALYSIS**
Based on the data provided, here is a concise analysis:

**1. Market Sen

'\n💰 **PRICE DATA**\nCurrent Price: $111,034.20\n24h Change: +2.22%\nVolume: 15,434\nData Source: Binance\n\n📈 **TECHNICAL INDICATORS**\n\nRSI (14): 45.11\nMACD: -1583.4439\nPrice vs SMA(20): -1.59%\nPrice vs SMA(50): N/A\n\n\n🤖 **AI ANALYSIS**\nBased on the data provided, here is a concise analysis:\n\n**1. Market Sentiment:** Neutral to slightly bearish. The positive 24h price change is countered by weak momentum indicators.\n\n**2. Key Technical Signals:** The RSI (45.11) indicates neutral momentum with a slight bearish tilt. The significantly negative MACD strongly suggests bearish momentum is present. Trading below the SMA(20) confirms a short-term downtrend.\n\n**3. Trading Recommendation:** **HOLD / SELL**. Confidence Level: Medium. The negative MACD and position below the short-term moving average advise against new long positions. Current holders should consider tightening stop-losses or taking profits.\n\n**4. Risk Assessment:** High. The low volume (15,434) relative to the h

In [9]:
# Test Ethereum
process_crypto_query_with_ai("Should I buy Ethereum now?")

🤖 Processing query: 'Should I buy Ethereum now?'
1️⃣ Extracting cryptocurrency ticker...
   ✅ Identified ticker: ETH

2️⃣ Fetching current price data...
   ✅ Current price: $4,341.99 (+1.53%)
   📊 Data source: binance

3️⃣ Performing technical analysis...
📊 Fetching ETH data from Binance API...
✅ Successfully fetched 30 records from Binance
🔄 Calculating technical indicators...
✅ Moving averages calculated
✅ Trend indicators calculated
✅ Momentum indicators calculated
✅ Volatility indicators calculated
✅ Volume indicators calculated
✅ All technical indicators calculated successfully!
   ✅ Technical analysis completed (30 records)

4️⃣ Generating AI analysis...

📊 ANALYSIS REPORT FOR ETH

💰 **PRICE DATA**
Current Price: $4,341.99
24h Change: +1.53%
Volume: 473,299
Data Source: Binance

📈 **TECHNICAL INDICATORS**

RSI (14): 50.81
MACD: 71.3944
Price vs SMA(20): -2.18%
Price vs SMA(50): N/A


🤖 **AI ANALYSIS**
Based on the provided data from Binance:

**1. Market Sentiment:** Neutral to s

'\n💰 **PRICE DATA**\nCurrent Price: $4,341.99\n24h Change: +1.53%\nVolume: 473,299\nData Source: Binance\n\n📈 **TECHNICAL INDICATORS**\n\nRSI (14): 50.81\nMACD: 71.3944\nPrice vs SMA(20): -2.18%\nPrice vs SMA(50): N/A\n\n\n🤖 **AI ANALYSIS**\nBased on the provided data from Binance:\n\n**1. Market Sentiment:** Neutral to slightly bullish. The positive 24h price change and RSI near 50 indicate balanced buying and selling pressure.\n\n**2. Key Signals:**\n*   The high MACD value (71.39) is a strong bullish momentum signal.\n*   Price trading slightly below the SMA(20) suggests minor short-term resistance.\n*   Strong volume supports the current price move.\n\n**3. Trading Recommendation:** **BUY** with Moderate Confidence. The powerful MACD signal outweighs the short-term SMA resistance, indicating potential upward momentum.\n\n**4. Risk Assessment:** Medium. The RSI is neutral, providing room for movement, but the lack of a SMA(50) comparison limits the view of the longer-term trend. Mon

In [10]:
# Test Solana
process_crypto_query_with_ai("What's the outlook for Solana?")

🤖 Processing query: 'What's the outlook for Solana?'
1️⃣ Extracting cryptocurrency ticker...
   ✅ Identified ticker: SOL

2️⃣ Fetching current price data...
   ✅ Current price: $208.88 (+5.62%)
   📊 Data source: binance

3️⃣ Performing technical analysis...
📊 Fetching SOL data from Binance API...
✅ Successfully fetched 30 records from Binance
🔄 Calculating technical indicators...
✅ Moving averages calculated
✅ Trend indicators calculated
✅ Momentum indicators calculated
✅ Volatility indicators calculated
✅ Volume indicators calculated
✅ All technical indicators calculated successfully!
   ✅ Technical analysis completed (30 records)

4️⃣ Generating AI analysis...

📊 ANALYSIS REPORT FOR SOL

💰 **PRICE DATA**
Current Price: $208.88
24h Change: +5.62%
Volume: 5,305,150
Data Source: Binance

📈 **TECHNICAL INDICATORS**

RSI (14): 59.27
MACD: 7.4695
Price vs SMA(20): +6.32%
Price vs SMA(50): N/A


🤖 **AI ANALYSIS**
Based on the provided data, the current market sentiment for SOL is **bullish*

'\n💰 **PRICE DATA**\nCurrent Price: $208.88\n24h Change: +5.62%\nVolume: 5,305,150\nData Source: Binance\n\n📈 **TECHNICAL INDICATORS**\n\nRSI (14): 59.27\nMACD: 7.4695\nPrice vs SMA(20): +6.32%\nPrice vs SMA(50): N/A\n\n\n🤖 **AI ANALYSIS**\nBased on the provided data, the current market sentiment for SOL is **bullish**.\n\nKey technical signals support this: a strong **+5.62%** 24h gain on high Binance volume indicates solid buying pressure. The **RSI at 59.27** is in healthy bullish territory without being overbought. The positive **MACD (7.4695)** and price trading **+6.32% above its SMA(20)** both confirm the short-term upward momentum.\n\n**Trading Recommendation: HOLD** for existing positions. Consider a **BUY** on a slight pullback for new entries. Confidence level is **Moderate-High**, as the trend is strong but the lack of a 50-day SMA comparison limits a full perspective.\n\n**Risk Assessment:** The main risk is a momentum reversal after a significant daily gain. The absence o

## 📝 Setup Instructions

### To use the AI features, you need:

1. **OpenRouter API Key** (for DeepSeek AI):
   - Sign up at https://openrouter.ai/
   - Get your API key
   - Add to `.env` file: `OPENROUTER_API_KEY=your_key_here`

2. **Virtual Environment**:
   ```bash
   source .venv/bin/activate
   jupyter notebook crypto_ai_analysis.ipynb
   ```

### Features:
- ✅ **AI Ticker Extraction**: Understands natural language queries
- ✅ **Real-time Prices**: Binance API with yfinance fallback
- ✅ **Technical Analysis**: 32+ indicators (RSI, MACD, etc.)
- ✅ **AI Analysis**: DeepSeek-powered trading recommendations
- ✅ **Complete Pipeline**: From question to actionable insights

### Example Queries:
- "How is Bitcoin performing today?"
- "Should I buy Ethereum now?"
- "What's happening with Solana?"
- "Is Cardano a good investment?"
- "Tell me about Polygon MATIC price"